# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pyspark.sql import SparkSession

# Create a local Spark session using all available cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_Test") \
    .getOrCreate()

# Verify the session
print(spark)



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import HfApi
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)



from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance",streaming = True, split="train")
df = ds.to_pandas()
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
con.sql("SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [12]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Split design check — time-aware, per-client cutoff (not one global date)

fact_rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
clients_rel = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

# 1. Get each client's own history start, from dim_clients (only clients with real GSC access)
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start
    FROM read_parquet('{clients_rel}')
    WHERE has_gsc_access IS TRUE
""").df()

# 2. Snapshot end date
snapshot_end = con.sql(f"""
    SELECT MAX(report_date) AS max_date
    FROM read_parquet('{fact_rel}')
""").df().loc[0, "max_date"]
print(f"Snapshot end date: {snapshot_end}")

# 3. Per-client cutoff = midpoint of (gsc_data_start, snapshot_end)
clients["cutoff_date"] = clients["gsc_data_start"] + (snapshot_end - clients["gsc_data_start"]) / 2
print(clients[["client_hash_id", "gsc_data_start", "cutoff_date"]].head())

con.register("client_cutoffs", clients[["client_hash_id", "cutoff_date"]])

# 4. Row counts on each side of the PER-CLIENT cutoff — only rows with real GSC data
counts = con.sql(f"""
    SELECT
        SUM(CASE WHEN f.report_date < c.cutoff_date THEN 1 ELSE 0 END) AS feature_rows,
        SUM(CASE WHEN f.report_date >= c.cutoff_date THEN 1 ELSE 0 END) AS label_rows
    FROM read_parquet('{fact_rel}') f
    JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
""").df()
print(counts)

# 5. Sanity check: every content item has real GSC data on both sides of ITS client's cutoff
pairs_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_content_items,
        SUM(CASE WHEN has_before AND has_after THEN 1 ELSE 0 END) AS items_with_both
    FROM (
        SELECT
            f.content_hash_id,
            BOOL_OR(f.report_date < c.cutoff_date) AS has_before,
            BOOL_OR(f.report_date >= c.cutoff_date) AS has_after
        FROM read_parquet('{fact_rel}') f
        JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
        WHERE f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id
    )
""").df()
print(pairs_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Snapshot end date: 2026-06-30 00:00:00
            client_hash_id gsc_data_start         cutoff_date
0  client_04660893ae39614a            NaT                 NaT
1  client_06d356715a8ff3b6     2026-04-10 2026-05-20 12:00:00
2  client_08a6a72ff48e62c0     2025-09-24 2026-02-10 12:00:00
3  client_08d2847f24cf89c1     2025-07-21 2026-01-09 00:00:00
4  client_0b245132bb722950     2026-04-12 2026-05-21 12:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   feature_rows  label_rows
0     8950079.0  19504192.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_content_items  items_with_both
0               297494         167964.0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.